# 前向传播（Forward Propagation）

用 NumPy 从零实现神经网络的前向传播：单层神经元计算 → 向量化优化 → 多层网络串联。

- 单层公式：`a_out = sigmoid(a_in @ W + b)`（Dense Layer）
- 示例层：3 个神经元，权重向量 w₁ = [1,2]、w₂ = [-3,4]、w₃ = [5,-6]，偏置 b₁=1、b₂=1、b₃=3


## 1. 神经网络单层（Dense Layer）的数学定义

一个神经元做的事：

$$
z = w·a_{in} + b, \quad a_{out} = sigmoid(z)
$$

3 个神经元拼成一个层，权重按列拼成矩阵 $W$（2 行 = 输入特征数，3 列 = 神经元数），偏置拼成向量 $B$：

$$
W = \begin{bmatrix} 1 & -3 & 5 \\ 2 & 4 & -6 \end{bmatrix}, \quad B = [1, 1, 3]
$$

输入 $a_{in} = [1, 2]$，整层输出就是各神经元输出的拼接：

$$
a_{out} = sigmoid(a_{in}·W + B) = [sigmoid(w_1·a_{in}+b_1),\; sigmoid(w_2·a_{in}+b_2),\; sigmoid(w_3·a_{in}+b_3)]
$$


In [ ]:
import numpy as np

# 权重矩阵：W[:, i] 是第 i 个神经元的权重向量（2 个输入特征 → 3 个神经元）
W = np.array([
    [1, -3, 5],
    [2, 4, -6]
])

# 输入样本（1 个样本、2 个特征）
a_in = np.array([1, 2])

# 偏置向量
b = np.array([1, 1, 3])

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

## 2. 逐神经元循环计算（理解原理）

最直观的写法：**逐个神经元**计算 `z = a_in · w_i + b_i`，再套 sigmoid。

伪代码：

```
for 每个神经元 i:
    z_i      = a_in · W[:, i] + b[i]   # 加权和 + 偏置
    a_out[i] = sigmoid(z_i)
```

缺点：Python 循环逐个计算，数据量大时很慢。


In [ ]:
# 循环版：逐个神经元计算
def dense_loop(a_in, W, b):
    # 支持单样本 (n_features,) 和批量 (n_samples, n_features)
    single = a_in.ndim == 1
    A = a_in[None, :] if single else a_in
    a_out = np.zeros((A.shape[0], W.shape[1]))   # (样本数, 神经元数)
    for i in range(W.shape[1]):
        z = A @ W[:, i] + b[i]                   # 第 i 个神经元的加权和（整批）
        a_out[:, i] = sigmoid(z)
    return a_out[0] if single else a_out

print('dense_loop 结果:', dense_loop(a_in, W, b))

## 3. 向量化计算（高效实现）

`a_in·W + B` 一次矩阵乘法就同时算完 3 个神经元的加权和，再整体套 sigmoid：

$$
a_{out} = sigmoid(a_{in}·W + B)
$$

> 矩阵乘法 $a_{in}·W$ 的第 i 列恰好就是第 i 个神经元的 $w_i·a_{in}$，所以**循环版和向量化版数学上完全等价**，只是向量化交给底层 BLAS 并行计算，速度大幅提升。


In [ ]:
# 向量化版：一次矩阵乘法 + 广播加法
def dense(a_in, W, b):
    z = np.matmul(a_in, W) + b   # b 自动广播到每个神经元
    a_out = sigmoid(z)
    return a_out

print('dense 向量化结果:', dense(a_in, W, b))

## 4. 两种方式对比

- **结果一致性**：单样本上两种方法输出应完全相同
- **效率**：批量数据下向量化明显更快


In [ ]:
# 结果一致性
assert np.allclose(dense_loop(a_in, W, b), dense(a_in, W, b))
print('两种方法结果一致 ✓')

# 批量数据下对比耗时（2000 样本 × 100 特征 → 200 个神经元）
import time
X = np.random.randn(2000, 100)
W_big = np.random.randn(100, 200)
b_big = np.random.randn(200)

t0 = time.time()
out_loop = dense_loop(X, W_big, b_big)   # 循环版
t1 = time.time()
print(f'循环版耗时: {t1 - t0:.4f}s')

t2 = time.time()
out_vec = dense(X, W_big, b_big)         # 向量化版
t3 = time.time()
print(f'向量化耗时: {t3 - t2:.4f}s')

## 5. 多层神经网络（Sequential）前向传播

多层网络就是把**上一层的输出当作下一层的输入**，逐层串联：

```
def sequential(x):
    a1 = dense(x,  W1, b1)   # 第 1 层：输入 → 隐藏层
    a2 = dense(a1, W2, b2)   # 第 2 层：隐藏层 → 输出层
    return a2
```

示例：2 个输入特征 → 3 个隐藏神经元 → 1 个输出神经元。


In [ ]:
# 第 1 层：2 输入 → 3 隐藏神经元
W1 = np.array([[1, -1, 2],
               [0,  1, -1]])
b1 = np.array([0, 1, -1])

# 第 2 层：3 隐藏 → 1 输出
W2 = np.array([[2], [-1], [1]])
b2 = np.array([0.5])

def sequential(x):
    a1 = dense(x, W1, b1)
    a2 = dense(a1, W2, b2)
    return a2

x = np.array([1, 2])
print('隐藏层输出 a1:', dense(x, W1, b1))
print('最终输出 fx :', sequential(x))